# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ku-ro-wa/flyrank-ml-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

##1.1 Finding One: The 469.9M impressions claim
This first claim comes from a repeated figure of 469,879,632 search impressions established multiple times in the report. The report states that the numbers are taken from a rolling 90 day window, but doesn't explicitly state a window of time.

This is confirmed by the fact that:
- The trailing 90 days from the most recent data (through 2026-06-30) =  ~764.3M impressions
- The best single 90-day rolling window across the entire history (any end date, daily granularity) = ~468.5M, ending 2026-03-06 — off by 1.4M (0.3%).
- Closest weekly-anchored window (from windows.parquet) = 472.8M, anchor 2026-03-30 — off by 2.9M (0.6%).

It would be beneficial to know the exact window of time the report is based on to accurately date it since the dataset's reported metrics have continued to grow as evidenced by the increase in impressions from the latest 90 days of recorded data compared to the stated figure. It may also be that the 90 days is sourced from another filter than just calendar dates, which would also be worth knowing about. And for transparency reasons should anyone wish to validate or audit the report.


##1.2 Finding Two: The ambiguitiy of 'scroll depth'
This second claim comes from another figure repeated throughout the report. This time this figure is the Health Score, calculated as follows:
$$Impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts)$$

Checking the provided data dictionary gives:
- `scroll_events_90d`: GA4 scroll events
- `scroll_rate` (a derived rate, x100%, and can exceed 100 (multiple scroll events per pageview)) = `scroll_events_90d / pageviews_90d * 100`

`scroll_events_90d` can be assumed to be an aggregated value made from a rolling 90 day window (from `scroll_events` in the original `fact_content_daily_performance table)` and `pageviews_90d` derived the same way.

There is no mention of any scroll depth value in the dictionary and no easy equivalent in any of the 5 tables unlike other components of the Health Score (`gsc_impressions, gsc_sum_position, gsc_avg_position, gsc_clicks, ctr` can be derived from this). Scroll depth is mentioned numerous times in the report while scroll rate is largely absent from the report save for a feature correlation heatmap (Page 25) where it is then immediately referred to as the former term below the heatmap.

This leads me to believe that the report either treats both terms (scroll depth and `scroll_rate`) as the same thing, which wouldn't make sense given that the latter can exceed 100 which means that `scroll_rate` is a per-pageview event count (potentially several threshold-crossings summed), not a per-pageview depth reading. Or that scroll depth refers to something else entirely that isn't defined in either the dictionary or report. Both of these outcomes aren't exactly ideal for the Health Score's transparency and would benefit from a clear definition / explanation.

#1.5 Environment: Self-contained setup
Standalone notebook but pulls from the previous notebook's setup and hurdle model for setup and evaluation purposes.

**Prerequisites**: Colab Pro/Pro+ High-RAM runtime; your HF account needs approved access to the gated `FlyRank/internship-warehouse` dataset

In [1]:
# Setup
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN {HF_TOKEN})")
rel = "hf://datasets/FlyRank/internship-warehouse"

# Check schema
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 0").show()

# Basic count
con.sql(f"""
    SELECT COUNT(*) AS n_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").show()


┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┐
│  n_rows  │
│  int64   │
├──────────┤
│ 78835655 │
└──────────┘



In [2]:
!pip install -q duckdb lightgbm


In [3]:
from huggingface_hub import login

login(token=HF_TOKEN)

In [4]:
import os

os.makedirs("warehouse", exist_ok=True)


In [5]:
%%writefile warehouse/__init__.py


Writing warehouse/__init__.py


**Note: The following cells feature lengthy code and are minimised by default, expand them to see the source code.**

In [6]:
#@title Connection helper to read the gated dataset on hugging face via duckdb
%%writefile warehouse/hf_setup.py
"""Connection helper for reading the gated FlyRank/internship-warehouse
dataset directly off the Hugging Face Hub via duckdb.

Requires `hf auth login` to have been run already (reads the cached token
via huggingface_hub.get_token()). duckdb's native hf:// protocol errors
opaquely against gated repos, so this uses plain https resolve URLs with
an explicit Bearer-token HTTP secret instead.
"""
import os

import duckdb
from huggingface_hub import get_token

REPO = "FlyRank/internship-warehouse"
RESOLVE_BASE = f"https://huggingface.co/datasets/{REPO}/resolve/main"

MONTHS = [
    "2025-01", "2025-02", "2025-03", "2025-04", "2025-05", "2025-06",
    "2025-07", "2025-08", "2025-09", "2025-10", "2025-11", "2025-12",
    "2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06",
]
FACT_FILES = [
    f"{RESOLVE_BASE}/fact_content_daily_performance/month={m}/data_0.parquet"
    for m in MONTHS
]
FACT_FILES_SQL = "[" + ",".join(f"'{u}'" for u in FACT_FILES) + "]"

DIM_CLIENTS_URL = f"{RESOLVE_BASE}/dim_clients.parquet"
DIM_CONTENT_URL = f"{RESOLVE_BASE}/dim_content.parquet"
FACT_QUERY_90D_URL = f"{RESOLVE_BASE}/fact_content_query_90d.parquet"


def get_con() -> duckdb.DuckDBPyConnection:
    tok = get_token()
    if not tok:
        raise RuntimeError("No cached Hugging Face token found — run `hf auth login` first.")
    con = duckdb.connect()
    sql = (
        "CREATE SECRET hf_http (TYPE HTTP, EXTRA_HTTP_HEADERS MAP "
        "{'Authorization': 'Bearer " + tok + "'}, SCOPE 'https://huggingface.co')"
    )
    con.execute(sql)
    del sql, tok
    try:
        # cosmetic only — fails under ipykernel when ipywidgets isn't installed
        # (duckdb routes this through IPython's widget-based progress renderer
        # there), which shouldn't be fatal to getting a working connection.
        con.execute("SET enable_progress_bar=false")
    except Exception:
        pass
    con.execute(f"SET threads={os.cpu_count()}")
    return con

Writing warehouse/hf_setup.py


In [7]:
#@title Code that establishes and trains the models used in the previous notebook
%%writefile warehouse/train.py
"""Feature matrix construction, temporal/client-holdout splitting, and
LightGBM (Tweedie objective) training for the two 30d targets.

Exposes functions rather than a __main__ script — meant to be driven from
the notebook so the actual run and its output stay visible there.
"""
from dataclasses import dataclass, field
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

WINDOWS_PATH = str(Path(__file__).resolve().parent.parent / "data" / "windows.parquet")

TARGETS = ["target_gsc_clicks_30d", "target_ga4_sessions_30d"]

RAW_SESSION_COLS = [
    "f_sessions_organic_90d", "f_sessions_direct_90d", "f_sessions_referral_90d",
    "f_sessions_social_90d", "f_sessions_paid_90d", "f_sessions_ai_90d",
]
RAW_AI_COLS = [
    "f_ai_chatgpt_90d", "f_ai_perplexity_90d", "f_ai_gemini_90d",
    "f_ai_copilot_90d", "f_ai_claude_90d", "f_ai_meta_90d", "f_ai_other_90d",
]

NUMERIC_FEATURES = [
    "f_gsc_impressions_90d", "f_gsc_clicks_90d", "f_gsc_ctr_90d",
    "f_gsc_avg_position_90d", "f_gsc_pct_days_active_90d", "f_gsc_momentum_ratio",
    "f_ga4_pageviews_90d", "f_ga4_sessions_90d", "f_ga4_engaged_sessions_90d",
    "f_ga4_engagement_rate_90d", "f_ga4_avg_engagement_sec_90d",
    "f_ga4_pct_days_active_90d", "f_ga4_momentum_ratio",
    "f_sessions_total_90d",
    "word_count", "search_volume", "competition", "f_content_age_days",
] + [f"{c}_share" for c in RAW_SESSION_COLS] + [f"{c}_share" for c in RAW_AI_COLS]

CATEGORICAL_FEATURES = ["content_type", "main_intent", "competition_level"]


def load_raw(path: str = WINDOWS_PATH) -> pd.DataFrame:
    df = pd.read_parquet(path)
    df["anchor_date"] = pd.to_datetime(df["anchor_date"])
    df["content_created_date"] = pd.to_datetime(df["content_created_date"])
    return df


def _add_share_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    total_sessions = df["f_sessions_total_90d"].replace(0, np.nan)
    for c in RAW_SESSION_COLS:
        df[f"{c}_share"] = df[c] / total_sessions
    total_ai = df["f_sessions_ai_90d"].replace(0, np.nan)
    for c in RAW_AI_COLS:
        df[f"{c}_share"] = df[c] / total_ai
    return df


def build_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    df = _add_share_columns(df)
    for c in CATEGORICAL_FEATURES:
        df[c] = df[c].astype("category")
    for c in NUMERIC_FEATURES:
        if c in df.columns and pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].astype("float32")
    return df


@dataclass
class Splits:
    train_idx: np.ndarray
    test_idx: np.ndarray
    cutoff: pd.Timestamp
    test_start: pd.Timestamp
    holdout_clients: list = field(default_factory=list)


def temporal_split(df: pd.DataFrame, cutoff: str, gap_days: int = 30) -> Splits:
    cutoff = pd.Timestamp(cutoff)
    test_start = cutoff + pd.Timedelta(days=gap_days)
    train_idx = df.index[df["anchor_date"] <= cutoff].to_numpy()
    test_idx = df.index[df["anchor_date"] >= test_start].to_numpy()
    return Splits(train_idx=train_idx, test_idx=test_idx, cutoff=cutoff, test_start=test_start)


def pick_holdout_clients(df: pd.DataFrame, n_holdout: int = 20, seed: int = 0) -> list:
    """Stratified-by-size sample of clients to exclude entirely for the
    client-generalization diagnostic (secondary check, not primary metric)."""
    counts = df.groupby("client_hash_id").size().sort_values()
    clients = counts.index.to_numpy()
    # stratify: take every k-th client across the size-sorted list
    k = max(1, len(clients) // n_holdout)
    rng = np.random.default_rng(seed)
    strata = [clients[i:i + k] for i in range(0, len(clients), k)]
    holdout = [rng.choice(s) for s in strata if len(s) > 0][:n_holdout]
    return list(holdout)


class ClientTargetEncoder:
    """Target-encodes client_hash_id using ONLY the rows it's fit on
    (must be fit on train split alone to avoid leakage)."""

    def __init__(self, smoothing: float = 10.0):
        self.smoothing = smoothing
        self.global_mean_ = None
        self.map_ = None

    def fit(self, client_ids: pd.Series, y: pd.Series):
        self.global_mean_ = y.mean()
        stats = y.groupby(client_ids).agg(["mean", "count"])
        self.map_ = (
            (stats["mean"] * stats["count"] + self.global_mean_ * self.smoothing)
            / (stats["count"] + self.smoothing)
        )
        return self

    def transform(self, client_ids: pd.Series) -> pd.Series:
        return client_ids.map(self.map_).fillna(self.global_mean_).astype("float32")


def _feature_cols_for_matrix(df: pd.DataFrame) -> list:
    return NUMERIC_FEATURES + CATEGORICAL_FEATURES + ["client_target_enc"]


# volume-type features where "more history" should never predict "less future",
# all else equal — constrained non-decreasing so tree splits can't reverse the
# ordering a plain linear scaling of these columns would preserve for free.
MONOTONE_INCREASING = {
    "f_gsc_clicks_90d", "f_gsc_impressions_90d",
    "f_ga4_sessions_90d", "f_ga4_pageviews_90d", "f_sessions_total_90d",
    "client_target_enc",
}


def _monotone_constraints(feature_cols: list) -> list:
    return [1 if c in MONOTONE_INCREASING else 0 for c in feature_cols]


def prepare_target_dataset(df: pd.DataFrame, target_col: str, exclude_clients: list = None):
    """Row-level target validity filtering, per target:
      - gsc: client must have GSC access (target never null in practice, but
        filter defensively).
      - ga4: drop rows where the client had zero GA4 access during the
        target window (structurally invalid, not sparse), and require
        >=25 of 30 target days had GA4 access (avoid partial-window bias
        from mid-window onboarding transitions).
    """
    d = df
    if target_col == "target_gsc_clicks_30d":
        d = d[d["client_has_gsc"] & d[target_col].notna()]
    elif target_col == "target_ga4_sessions_30d":
        d = d[(d["target_ga4_access_days_30d"] >= 25) & d[target_col].notna()]
    if exclude_clients:
        d = d[~d["client_hash_id"].isin(exclude_clients)]
    return d


def make_lgb_datasets(df: pd.DataFrame, target_col: str, splits: Splits,
                       exclude_clients: list = None, val_weeks: int = 8):
    d = prepare_target_dataset(df, target_col, exclude_clients=exclude_clients)
    train_all = d.loc[d.index.intersection(splits.train_idx)]
    test = d.loc[d.index.intersection(splits.test_idx)]

    val_start = splits.cutoff - pd.Timedelta(weeks=val_weeks)
    tr = train_all[train_all["anchor_date"] < val_start].copy()
    val = train_all[train_all["anchor_date"] >= val_start].copy()
    test = test.copy()

    enc = ClientTargetEncoder().fit(tr["client_hash_id"], tr[target_col])
    for part in (tr, val, test):
        part["client_target_enc"] = enc.transform(part["client_hash_id"])

    cols = _feature_cols_for_matrix(d)
    cat_idx = [cols.index(c) for c in CATEGORICAL_FEATURES]

    ds_tr = lgb.Dataset(tr[cols], label=tr[target_col], categorical_feature=cat_idx)
    ds_val = lgb.Dataset(val[cols], label=val[target_col], categorical_feature=cat_idx, reference=ds_tr)

    return dict(train=tr, val=val, test=test, ds_train=ds_tr, ds_val=ds_val,
                feature_cols=cols, encoder=enc)


def train_tweedie_model(data: dict, tweedie_variance_power: float = 1.5,
                         num_boost_round: int = 3000, early_stopping_rounds: int = 100,
                         learning_rate: float = 0.05, num_leaves: int = 63,
                         min_data_in_leaf: int = 100):
    params = dict(
        objective="tweedie",
        tweedie_variance_power=tweedie_variance_power,
        metric="tweedie",
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        min_data_in_leaf=min_data_in_leaf,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        bagging_freq=5,
        monotone_constraints=_monotone_constraints(data["feature_cols"]),
        verbose=-1,
    )
    model = lgb.train(
        params, data["ds_train"],
        num_boost_round=num_boost_round,
        valid_sets=[data["ds_val"]],
        callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False), lgb.log_evaluation(0)],
    )
    return model


def make_client_holdout_datasets(df: pd.DataFrame, target_col: str, splits: Splits,
                                  holdout_clients: list, val_weeks: int = 8):
    """Train/val exclude the held-out clients entirely (never seen);
    the returned 'test_holdout' set is the temporal-test-period rows for
    exactly those clients, so it's directly comparable to how the primary
    (full-data) model scores on the same rows — the delta between the two
    quantifies reliance on client-specific memorization vs transferable signal.
    """
    d = prepare_target_dataset(df, target_col)
    train_all = d.loc[d.index.intersection(splits.train_idx)]
    train_all = train_all[~train_all["client_hash_id"].isin(holdout_clients)]
    test_holdout = d.loc[d.index.intersection(splits.test_idx)]
    test_holdout = test_holdout[test_holdout["client_hash_id"].isin(holdout_clients)]

    val_start = splits.cutoff - pd.Timedelta(weeks=val_weeks)
    tr = train_all[train_all["anchor_date"] < val_start].copy()
    val = train_all[train_all["anchor_date"] >= val_start].copy()
    test_holdout = test_holdout.copy()

    enc = ClientTargetEncoder().fit(tr["client_hash_id"], tr[target_col])
    for part in (tr, val, test_holdout):
        part["client_target_enc"] = enc.transform(part["client_hash_id"])

    cols = _feature_cols_for_matrix(d)
    cat_idx = [cols.index(c) for c in CATEGORICAL_FEATURES]
    ds_tr = lgb.Dataset(tr[cols], label=tr[target_col], categorical_feature=cat_idx)
    ds_val = lgb.Dataset(val[cols], label=val[target_col], categorical_feature=cat_idx, reference=ds_tr)

    return dict(train=tr, val=val, test_holdout=test_holdout, ds_train=ds_tr, ds_val=ds_val,
                feature_cols=cols, encoder=enc)


def tune_tweedie_power(data: dict, candidates=(1.1, 1.3, 1.5, 1.7, 1.9)):
    best = None
    for p in candidates:
        m = train_tweedie_model(data, tweedie_variance_power=p)
        score = m.best_score["valid_0"]["tweedie"]
        if best is None or score < best[1]:
            best = (p, score, m)
    return best  # (power, val_score, model)


# ---------------------------------------------------------------------------
# Iteration 2: two-part / hurdle model (classify "any activity" + regress
# conditional on activity), and a general hyperparameter random search.
# ---------------------------------------------------------------------------

def make_hurdle_datasets(df: pd.DataFrame, target_col: str, splits: Splits,
                          exclude_clients: list = None, val_weeks: int = 8):
    """Like make_lgb_datasets, but returns two dataset dicts: 'clf' (label =
    target_col > 0, all valid rows) and 'reg' (label = target_col, restricted
    to rows where target_col > 0). Both stages share one ClientTargetEncoder
    fit on the full (pre-activity-filter) train split so the encoding is
    consistent between them."""
    d = prepare_target_dataset(df, target_col, exclude_clients=exclude_clients)
    train_all = d.loc[d.index.intersection(splits.train_idx)]
    test = d.loc[d.index.intersection(splits.test_idx)]

    val_start = splits.cutoff - pd.Timedelta(weeks=val_weeks)
    tr = train_all[train_all["anchor_date"] < val_start].copy()
    val = train_all[train_all["anchor_date"] >= val_start].copy()
    test = test.copy()

    enc = ClientTargetEncoder().fit(tr["client_hash_id"], tr[target_col])
    for part in (tr, val, test):
        part["client_target_enc"] = enc.transform(part["client_hash_id"])

    cols = _feature_cols_for_matrix(d)
    cat_idx = [cols.index(c) for c in CATEGORICAL_FEATURES]

    # feature_pre_filter=False: these datasets are reused across many
    # random-search trials with varying min_data_in_leaf, and LightGBM's
    # default pre-filter caches feature-usability decisions from whichever
    # min_data_in_leaf the Dataset first constructs with — silently making
    # later trials with a smaller min_data_in_leaf wrong instead of erroring.
    ds_params = {"feature_pre_filter": False}

    def _clf_ds(part):
        return lgb.Dataset(part[cols], label=(part[target_col] > 0).astype(int),
                            categorical_feature=cat_idx, params=ds_params)

    ds_tr_clf = _clf_ds(tr)
    ds_val_clf = lgb.Dataset(val[cols], label=(val[target_col] > 0).astype(int),
                              categorical_feature=cat_idx, reference=ds_tr_clf, params=ds_params)
    clf_data = dict(train=tr, val=val, test=test, ds_train=ds_tr_clf, ds_val=ds_val_clf,
                     feature_cols=cols, encoder=enc)

    tr_act, val_act = tr[tr[target_col] > 0], val[val[target_col] > 0]
    ds_tr_reg = lgb.Dataset(tr_act[cols], label=tr_act[target_col],
                             categorical_feature=cat_idx, params=ds_params)
    ds_val_reg = lgb.Dataset(val_act[cols], label=val_act[target_col],
                              categorical_feature=cat_idx, reference=ds_tr_reg, params=ds_params)
    reg_data = dict(train=tr_act, val=val_act, test=test, ds_train=ds_tr_reg, ds_val=ds_val_reg,
                     feature_cols=cols, encoder=enc)

    return dict(clf=clf_data, reg=reg_data, test=test, feature_cols=cols, encoder=enc)


def make_hurdle_holdout_datasets(df: pd.DataFrame, target_col: str, splits: Splits,
                                  holdout_clients: list, val_weeks: int = 8):
    """Client-holdout counterpart of make_hurdle_datasets: train/val exclude
    the held-out clients entirely; test_holdout is the temporal-test-period
    rows for exactly those clients (shared across both stages)."""
    d = prepare_target_dataset(df, target_col)
    train_all = d.loc[d.index.intersection(splits.train_idx)]
    train_all = train_all[~train_all["client_hash_id"].isin(holdout_clients)]
    test_holdout = d.loc[d.index.intersection(splits.test_idx)]
    test_holdout = test_holdout[test_holdout["client_hash_id"].isin(holdout_clients)]

    val_start = splits.cutoff - pd.Timedelta(weeks=val_weeks)
    tr = train_all[train_all["anchor_date"] < val_start].copy()
    val = train_all[train_all["anchor_date"] >= val_start].copy()
    test_holdout = test_holdout.copy()

    enc = ClientTargetEncoder().fit(tr["client_hash_id"], tr[target_col])
    for part in (tr, val, test_holdout):
        part["client_target_enc"] = enc.transform(part["client_hash_id"])

    cols = _feature_cols_for_matrix(d)
    cat_idx = [cols.index(c) for c in CATEGORICAL_FEATURES]

    # feature_pre_filter=False: these datasets are reused across many
    # random-search trials with varying min_data_in_leaf, and LightGBM's
    # default pre-filter caches feature-usability decisions from whichever
    # min_data_in_leaf the Dataset first constructs with — silently making
    # later trials with a smaller min_data_in_leaf wrong instead of erroring.
    ds_params = {"feature_pre_filter": False}

    def _clf_ds(part):
        return lgb.Dataset(part[cols], label=(part[target_col] > 0).astype(int),
                            categorical_feature=cat_idx, params=ds_params)

    ds_tr_clf = _clf_ds(tr)
    ds_val_clf = lgb.Dataset(val[cols], label=(val[target_col] > 0).astype(int),
                              categorical_feature=cat_idx, reference=ds_tr_clf, params=ds_params)
    clf_data = dict(train=tr, val=val, test_holdout=test_holdout, ds_train=ds_tr_clf, ds_val=ds_val_clf,
                     feature_cols=cols, encoder=enc)

    tr_act, val_act = tr[tr[target_col] > 0], val[val[target_col] > 0]
    ds_tr_reg = lgb.Dataset(tr_act[cols], label=tr_act[target_col],
                             categorical_feature=cat_idx, params=ds_params)
    ds_val_reg = lgb.Dataset(val_act[cols], label=val_act[target_col],
                              categorical_feature=cat_idx, reference=ds_tr_reg, params=ds_params)
    reg_data = dict(train=tr_act, val=val_act, test_holdout=test_holdout, ds_train=ds_tr_reg, ds_val=ds_val_reg,
                     feature_cols=cols, encoder=enc)

    return dict(clf=clf_data, reg=reg_data, test_holdout=test_holdout, feature_cols=cols, encoder=enc)


def train_classifier(data: dict, num_leaves: int = 63, learning_rate: float = 0.05,
                      min_data_in_leaf: int = 100, feature_fraction: float = 0.8,
                      lambda_l1: float = 0.0, lambda_l2: float = 0.0,
                      num_boost_round: int = 2000, early_stopping_rounds: int = 100):
    params = dict(
        objective="binary",
        metric="binary_logloss",
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        min_data_in_leaf=min_data_in_leaf,
        feature_fraction=feature_fraction,
        bagging_fraction=0.8,
        bagging_freq=5,
        lambda_l1=lambda_l1,
        lambda_l2=lambda_l2,
        feature_pre_filter=False,
        monotone_constraints=_monotone_constraints(data["feature_cols"]),
        verbose=-1,
    )
    model = lgb.train(
        params, data["ds_train"],
        num_boost_round=num_boost_round,
        valid_sets=[data["ds_val"]],
        callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False), lgb.log_evaluation(0)],
    )
    return model


def train_regressor_conditional(data: dict, tweedie_variance_power: float = 1.5,
                                 num_leaves: int = 63, learning_rate: float = 0.05,
                                 min_data_in_leaf: int = 100, feature_fraction: float = 0.8,
                                 lambda_l1: float = 0.0, lambda_l2: float = 0.0,
                                 num_boost_round: int = 2000, early_stopping_rounds: int = 100):
    params = dict(
        objective="tweedie",
        tweedie_variance_power=tweedie_variance_power,
        metric="tweedie",
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        min_data_in_leaf=min_data_in_leaf,
        feature_fraction=feature_fraction,
        bagging_fraction=0.8,
        bagging_freq=5,
        lambda_l1=lambda_l1,
        lambda_l2=lambda_l2,
        feature_pre_filter=False,
        monotone_constraints=_monotone_constraints(data["feature_cols"]),
        verbose=-1,
    )
    model = lgb.train(
        params, data["ds_train"],
        num_boost_round=num_boost_round,
        valid_sets=[data["ds_val"]],
        callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False), lgb.log_evaluation(0)],
    )
    return model


class HurdleModel:
    """P(active) * E[value | active]. Exposes .predict(X) so it's a drop-in
    replacement anywhere a lgb.Booster is used (evaluate.py needs no changes)."""

    def __init__(self, clf, reg):
        self.clf = clf
        self.reg = reg

    def predict(self, X):
        p_active = np.clip(self.clf.predict(X), 0, 1)
        e_active = np.clip(self.reg.predict(X), 0, None)
        return p_active * e_active


def _random_search(train_fn, data: dict, param_space: dict, metric_key: str,
                    n_trials: int = 15, seed: int = 0, higher_is_better: bool = False):
    """Samples n_trials random hyperparameter combos from param_space (dict of
    name -> list of candidate values), trains via train_fn(data, **params),
    and picks the combo with the best data['ds_val'] metric_key score (lowest
    by default; set higher_is_better=True for metrics like NDCG). Returns
    (best_params, best_score, best_model)."""
    rng = np.random.default_rng(seed)
    keys = list(param_space)
    best = None
    for _ in range(n_trials):
        params = {k: rng.choice(param_space[k]).item() for k in keys}
        model = train_fn(data, **params)
        score = model.best_score["valid_0"][metric_key]
        is_better = best is None or (score > best[1] if higher_is_better else score < best[1])
        if is_better:
            best = (params, score, model)
    return best  # (params, val_score, model)


HURDLE_PARAM_SPACE = dict(
    num_leaves=[15, 31, 63, 127],
    learning_rate=[0.02, 0.03, 0.05, 0.08, 0.1],
    min_data_in_leaf=[20, 50, 100, 200, 400],
    feature_fraction=[0.6, 0.7, 0.8, 0.9, 1.0],
    lambda_l1=[0.0, 0.1, 1.0],
    lambda_l2=[0.0, 0.1, 1.0],
)


def tune_classifier(data: dict, n_trials: int = 15, seed: int = 0, param_space: dict = None):
    return _random_search(train_classifier, data, param_space or HURDLE_PARAM_SPACE,
                           metric_key="binary_logloss", n_trials=n_trials, seed=seed)


def tune_regressor_conditional(data: dict, tweedie_variance_power: float = 1.5,
                                n_trials: int = 15, seed: int = 0, param_space: dict = None):
    def _train_fn(d, **params):
        return train_regressor_conditional(d, tweedie_variance_power=tweedie_variance_power, **params)
    return _random_search(_train_fn, data, param_space or HURDLE_PARAM_SPACE,
                           metric_key="tweedie", n_trials=n_trials, seed=seed)


# ---------------------------------------------------------------------------
# Iteration 4a: segmented serving rule (route each activity bucket to
# whichever of {model, naive} wins there on validation).
# ---------------------------------------------------------------------------

DEFAULT_BUCKET_BINS = [0, 0.05, 0.25, 0.5, 0.75, 0.95, 1.0]


def fit_blend_router(val_df: pd.DataFrame, model, feature_cols: list, target_col: str,
                      activity_col: str, raw_feature_col: str, window_days: int = 90,
                      bins: list = None) -> dict:
    """Buckets val_df by activity_col and picks, per bucket, whichever of
    {model, naive-scale} is at least as good on ALL of MAE, RMSE, and Spearman
    on validation — not just MAE — before routing away from naive. Requiring
    agreement across metrics avoids routing on a single noisy signal that can
    flip between validation and test (see notebooks/model_training.ipynb
    Section 11 for the GSC (0.75, 0.95] bucket where MAE-only routing picked
    the model on val but naive actually won on test). Fit on val, not test —
    picking bucket winners from test results would be snooping."""
    bins = bins or DEFAULT_BUCKET_BINS
    d = val_df.copy()
    d["_pred_model"] = np.clip(model.predict(d[feature_cols]), 0, None)
    d["_pred_naive"] = NaiveModel(raw_feature_col, window_days).predict(d)
    d["_bucket"] = pd.cut(d[activity_col], bins=bins, include_lowest=True)

    router = {}
    for bucket, g in d.groupby("_bucket", observed=True):
        mae_model = mean_absolute_error(g[target_col], g["_pred_model"])
        mae_naive = mean_absolute_error(g[target_col], g["_pred_naive"])
        rmse_model = mean_squared_error(g[target_col], g["_pred_model"]) ** 0.5
        rmse_naive = mean_squared_error(g[target_col], g["_pred_naive"]) ** 0.5

        mae_ok = mae_model <= mae_naive
        rmse_ok = rmse_model <= rmse_naive
        if g[target_col].nunique() > 1:
            sp_model = spearmanr(g[target_col], g["_pred_model"]).statistic
            sp_naive = spearmanr(g[target_col], g["_pred_naive"]).statistic
            spearman_ok = sp_model >= sp_naive
        else:
            spearman_ok = True  # undefined when target is constant; don't block on it

        router[bucket] = "model" if (mae_ok and rmse_ok and spearman_ok) else "naive"
    return router


class NaiveModel:
    """Wraps the trivial 'scale prior window to 30d' baseline as a .predict(X)
    object so it plugs into score()/breakdown_by()/score_within_group() like
    every other model here, instead of needing bespoke inline code per use."""

    def __init__(self, raw_feature_col: str, window_days: int = 90):
        self.raw_feature_col = raw_feature_col
        self.window_days = window_days

    def predict(self, X):
        return np.clip(X[self.raw_feature_col].to_numpy() * (30 / self.window_days), 0, None)


class BlendedModel:
    """Routes each row to the model or the naive baseline based on which one
    won that row's activity bucket on validation (see fit_blend_router).
    Exposes .predict(X) — activity_col and raw_feature_col are already
    present in feature_cols, so no extra columns are needed beyond what
    score()/breakdown_by() already pass."""

    def __init__(self, model, router: dict, activity_col: str, raw_feature_col: str,
                 window_days: int = 90, bins: list = None):
        self.model = model
        self.router = router
        self.activity_col = activity_col
        self.raw_feature_col = raw_feature_col
        self.window_days = window_days
        self.bins = bins or DEFAULT_BUCKET_BINS

    def predict(self, X):
        pred_model = np.clip(self.model.predict(X), 0, None)
        pred_naive = NaiveModel(self.raw_feature_col, self.window_days).predict(X)
        bucket = pd.cut(X[self.activity_col], bins=self.bins, include_lowest=True)
        use_model = bucket.map(lambda b: self.router.get(b, "naive") == "model").to_numpy()
        return np.where(use_model, pred_model, pred_naive)


# ---------------------------------------------------------------------------
# Iteration 4b: learning-to-rank (lambdarank), grouped by
# (client_hash_id, anchor_date) — ranks content within a client's portfolio
# at a given point in time, rather than forecasting an absolute count.
# ---------------------------------------------------------------------------

def _relevance_labels(train_target: pd.Series, target: pd.Series, n_positive_bins: int = 4) -> np.ndarray:
    """0 for zero-activity rows; quartiles of the *positive* values (fit on
    train only) give levels 1..n_positive_bins for active rows."""
    positive_train = train_target[train_target > 0]
    edges = np.quantile(positive_train, np.linspace(0, 1, n_positive_bins + 1))
    edges = np.unique(edges)
    labels = np.zeros(len(target), dtype=int)
    positive_mask = (target > 0).to_numpy()
    if len(edges) > 2:
        labels[positive_mask] = np.clip(
            np.digitize(target[positive_mask], edges[1:-1], right=True) + 1, 1, n_positive_bins
        )
    else:
        labels[positive_mask] = 1
    return labels


def make_ranker_datasets(df: pd.DataFrame, target_col: str, splits: Splits,
                          val_weeks: int = 8, group_cols=("client_hash_id", "anchor_date")):
    d = prepare_target_dataset(df, target_col)
    train_all = d.loc[d.index.intersection(splits.train_idx)]
    test = d.loc[d.index.intersection(splits.test_idx)]

    val_start = splits.cutoff - pd.Timedelta(weeks=val_weeks)
    tr = train_all[train_all["anchor_date"] < val_start].copy()
    val = train_all[train_all["anchor_date"] >= val_start].copy()
    test = test.copy()

    enc = ClientTargetEncoder().fit(tr["client_hash_id"], tr[target_col])
    for part in (tr, val, test):
        part["client_target_enc"] = enc.transform(part["client_hash_id"])

    cols = _feature_cols_for_matrix(d)
    cat_idx = [cols.index(c) for c in CATEGORICAL_FEATURES]
    group_cols = list(group_cols)

    # LightGBM's lambdarank hard-caps a single query group at 10,000 rows —
    # some clients have far more content than that active on a single anchor
    # week, so oversized groups are deterministically split into <=8000-row
    # chunks (arbitrary positional split; there's no meaningful sub-ordering
    # to preserve, any partition is still a valid "rank within a subset of
    # this client's portfolio" unit).
    MAX_GROUP_SIZE = 8000

    def _sorted_grouped(part):
        part = part.sort_values(group_cols)
        raw_sizes = part.groupby(group_cols, sort=False).size().to_numpy()
        chunk_id = np.concatenate([np.arange(n) // MAX_GROUP_SIZE for n in raw_sizes])
        part = part.assign(_chunk_id=chunk_id)
        sub_group_cols = group_cols + ["_chunk_id"]
        part = part.sort_values(sub_group_cols)
        sizes = part.groupby(sub_group_cols, sort=False).size().to_numpy()
        part = part.drop(columns="_chunk_id")

        keep_groups = sizes >= 2
        if not keep_groups.all():
            # rebuild the row mask from the (already sorted, contiguous) group sizes
            row_mask = np.repeat(keep_groups, sizes)
            part = part[row_mask]
            sizes = sizes[keep_groups]
        return part, sizes

    tr, tr_sizes = _sorted_grouped(tr)
    val, val_sizes = _sorted_grouped(val)
    test, test_sizes = _sorted_grouped(test)

    tr_labels = _relevance_labels(tr[target_col], tr[target_col])
    val_labels = _relevance_labels(tr[target_col], val[target_col])
    test_labels = _relevance_labels(tr[target_col], test[target_col])

    ds_tr = lgb.Dataset(tr[cols], label=tr_labels, group=tr_sizes,
                         categorical_feature=cat_idx,
                         params={"feature_pre_filter": False})
    ds_val = lgb.Dataset(val[cols], label=val_labels, group=val_sizes,
                          categorical_feature=cat_idx, reference=ds_tr,
                          params={"feature_pre_filter": False})

    return dict(train=tr, val=val, test=test, ds_train=ds_tr, ds_val=ds_val,
                feature_cols=cols, encoder=enc, group_cols=group_cols)


def train_ranker(data: dict, num_leaves: int = 63, learning_rate: float = 0.05,
                  min_data_in_leaf: int = 100, feature_fraction: float = 0.8,
                  lambda_l1: float = 0.0, lambda_l2: float = 0.0,
                  num_boost_round: int = 2000, early_stopping_rounds: int = 100):
    params = dict(
        objective="lambdarank",
        metric="ndcg",
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        min_data_in_leaf=min_data_in_leaf,
        feature_fraction=feature_fraction,
        bagging_fraction=0.8,
        bagging_freq=5,
        lambda_l1=lambda_l1,
        lambda_l2=lambda_l2,
        feature_pre_filter=False,
        verbose=-1,
    )
    model = lgb.train(
        params, data["ds_train"],
        num_boost_round=num_boost_round,
        valid_sets=[data["ds_val"]],
        callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False), lgb.log_evaluation(0)],
    )
    return model


def tune_ranker(data: dict, n_trials: int = 8, seed: int = 0, param_space: dict = None):
    return _random_search(train_ranker, data, param_space or HURDLE_PARAM_SPACE,
                           metric_key="ndcg@1", n_trials=n_trials, seed=seed, higher_is_better=True)


Writing warehouse/train.py


In [8]:
#@title Metrics and error-breakdown helpers for the trained Tweedie models
%%writefile warehouse/evaluate.py
"""Metrics and error-breakdown helpers for the trained Tweedie models."""
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error


def tweedie_deviance(y_true, y_pred, p=1.5, eps=1e-8):
    y_pred = np.clip(y_pred, eps, None)
    y_true = np.asarray(y_true)
    # standard Tweedie deviance formula (p != 0,1,2 general case)
    a = np.power(y_true, 2 - p) / ((1 - p) * (2 - p))
    b = y_true * np.power(y_pred, 1 - p) / (1 - p)
    c = np.power(y_pred, 2 - p) / (2 - p)
    return float(np.mean(2 * (a - b + c)))


def score(model, X, y, tweedie_power=1.5) -> dict:
    pred = model.predict(X)
    pred = np.clip(pred, 0, None)
    return dict(
        n=len(y),
        mae=mean_absolute_error(y, pred),
        rmse=mean_squared_error(y, pred) ** 0.5,
        spearman=spearmanr(y, pred).statistic,
        tweedie_deviance=tweedie_deviance(y, pred, p=tweedie_power),
        mean_actual=float(np.mean(y)),
        mean_pred=float(np.mean(pred)),
    )


def breakdown_by(model, df: pd.DataFrame, feature_cols: list, target_col: str,
                  by: str, bins=None, tweedie_power=1.5) -> pd.DataFrame:
    d = df.copy()
    pred = np.clip(model.predict(d[feature_cols]), 0, None)
    d["_pred"] = pred
    if bins is not None:
        d["_bucket"] = pd.cut(d[by], bins=bins, include_lowest=True)
        group_col = "_bucket"
    else:
        group_col = by

    rows = []
    for key, g in d.groupby(group_col, observed=True):
        if len(g) < 5:
            continue
        rows.append(dict(
            group=key,
            n=len(g),
            mae=mean_absolute_error(g[target_col], g["_pred"]),
            rmse=mean_squared_error(g[target_col], g["_pred"]) ** 0.5,
            spearman=spearmanr(g[target_col], g["_pred"]).statistic if g[target_col].nunique() > 1 else np.nan,
            mean_actual=g[target_col].mean(),
            mean_pred=g["_pred"].mean(),
        ))
    return pd.DataFrame(rows).sort_values("n", ascending=False)


def baseline_naive_scale(df: pd.DataFrame, feature_90d_col: str, target_col: str, window_days: int = 90) -> dict:
    """Trivial baseline: scale the prior-window sum down to a 30d-equivalent
    (multiply by 30/window_days). Any real model should beat this."""
    pred = df[feature_90d_col].to_numpy() * (30 / window_days)
    pred = np.clip(pred, 0, None)
    y = df[target_col].to_numpy()
    return dict(
        n=len(y),
        mae=mean_absolute_error(y, pred),
        rmse=mean_squared_error(y, pred) ** 0.5,
        spearman=spearmanr(y, pred).statistic,
        mean_actual=float(np.mean(y)),
        mean_pred=float(np.mean(pred)),
    )


def score_within_group(model, df: pd.DataFrame, feature_cols: list, target_col: str,
                        group_cols=("client_hash_id", "anchor_date"), min_group_size: int = 2) -> dict:
    """Mean/median Spearman computed PER GROUP rather than pooled — this is
    what a lambdarank model (grouped the same way) is actually optimizing
    for. Pooled Spearman conflates within-group order with between-group
    scale differences a ranker isn't trying to get right, so this should be
    reported alongside pooled score(), not instead of it."""
    d = df.copy()
    # NOT clipped to >=0: a ranker's raw relevance score is unbounded and its
    # scale is meaningless, only order matters (Spearman is scale-invariant,
    # but clipping negatives to 0 creates artificial ties and would distort it).
    d["_pred"] = model.predict(d[feature_cols])
    group_cols = list(group_cols)

    per_group = []
    for _, g in d.groupby(group_cols, observed=True):
        if len(g) < min_group_size or g[target_col].nunique() <= 1:
            continue
        per_group.append(spearmanr(g[target_col], g["_pred"]).statistic)

    per_group = np.array(per_group, dtype=float)
    return dict(
        n_groups_scored=len(per_group),
        n_groups_total=d.groupby(group_cols, observed=True).ngroups,
        mean_within_group_spearman=float(np.nanmean(per_group)) if len(per_group) else float("nan"),
        median_within_group_spearman=float(np.nanmedian(per_group)) if len(per_group) else float("nan"),
    )


def compare_primary_vs_holdout(primary_model, holdout_model, holdout_test_df: pd.DataFrame,
                                feature_cols: list, target_col: str, tweedie_power=1.5) -> pd.DataFrame:
    """Both models scored on the SAME rows (test-period rows of the held-out
    clients) — primary saw these clients during training (just not these
    dates), holdout never saw them at all. The gap quantifies client-specific
    memorization vs transferable signal."""
    X = holdout_test_df[feature_cols]
    y = holdout_test_df[target_col]
    rows = {
        "primary_model (client seen in train)": score(primary_model, X, y, tweedie_power),
        "holdout_model (client never seen)": score(holdout_model, X, y, tweedie_power),
    }
    return pd.DataFrame(rows).T


Writing warehouse/evaluate.py


In [9]:
import sys
sys.path.insert(0, ".")

from warehouse import train, evaluate


### Download pre-built `data/windows.parquet`

Same file `w05_model_temp.ipynb` trains on, pulled from the private artifacts repo instead of rebuilt locally.


In [10]:
import shutil
from huggingface_hub import hf_hub_download

os.makedirs("data", exist_ok=True)

downloaded_path = hf_hub_download(
    repo_id="Ruo-ning/internship-warehouse-artifacts",
    filename="windows.parquet",
    repo_type="dataset",
    token=HF_TOKEN,
)
shutil.copy(downloaded_path, "data/windows.parquet")


windows.parquet: reconstructing file:   0%|          |  0.00B /  130MB            

windows.parquet: downloading bytes:           |  0.00B            

'data/windows.parquet'

In [11]:
raw = train.load_raw()
df = train.build_feature_matrix(raw)
df.shape


(5678150, 68)

In [12]:
SAMPLE_FRAC = 1.0  # manual fallback: lower this (e.g. 0.3) only if the cells
# below still hit Colab's free-tier memory ceiling after the optimizations in
# warehouse/train.py (Section 1). Trades exact fidelity to model_training.ipynb's
# numbers for a representative-sample approximation -- shouldn't change the
# qualitative conclusions (GA4 hurdle wins, GSC hurdle is mixed).
if SAMPLE_FRAC < 1.0:
    df = df.sample(frac=SAMPLE_FRAC, random_state=0)
    print(f"subsampled to {len(df):,} rows (SAMPLE_FRAC={SAMPLE_FRAC})")


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [13]:
import gc

import lightgbm as lgb
import numpy as np
import pandas as pd

# "Before": a deliberately naive split that a first pass at this problem might
# reach for -- pure random row shuffle, ignoring both anchor_date ordering and
# client grouping. Contrasts against train.temporal_split, the honest split
# w05_model_temp.ipynb actually used throughout.
def naive_random_split(df, test_frac=0.2, val_frac=0.1, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(df.index.to_numpy())
    n_test = int(len(idx) * test_frac)
    n_val = int(len(idx) * val_frac)
    test_idx = idx[:n_test]
    val_idx = idx[n_test:n_test + n_val]
    train_idx = idx[n_test + n_val:]
    return train_idx, val_idx, test_idx


# Mirrors train.make_hurdle_datasets' encoding/column logic, but driven off
# arbitrary index arrays instead of a train.Splits object -- make_hurdle_datasets
# hardcodes a time-based val carve-out via splits.cutoff, which doesn't apply
# to a split that ignores time. Only used for the naive "before" side; the
# "after" side reuses make_hurdle_datasets directly, unmodified.
def build_hurdle_from_idx(df, target_col, train_idx, val_idx, test_idx):
    d = train.prepare_target_dataset(df, target_col)
    tr = d.loc[d.index.intersection(train_idx)].copy()
    val = d.loc[d.index.intersection(val_idx)].copy()
    test = d.loc[d.index.intersection(test_idx)].copy()

    enc = train.ClientTargetEncoder().fit(tr["client_hash_id"], tr[target_col])
    for part in (tr, val, test):
        part["client_target_enc"] = enc.transform(part["client_hash_id"])

    cols = train.NUMERIC_FEATURES + train.CATEGORICAL_FEATURES + ["client_target_enc"]
    cat_idx = [cols.index(c) for c in train.CATEGORICAL_FEATURES]
    ds_params = {"feature_pre_filter": False}

    def _clf_ds(part, ref=None):
        return lgb.Dataset(part[cols], label=(part[target_col] > 0).astype(int),
                            categorical_feature=cat_idx, reference=ref, params=ds_params)

    ds_tr_clf = _clf_ds(tr)
    ds_val_clf = _clf_ds(val, ref=ds_tr_clf)
    clf_data = dict(train=tr, val=val, test=test, ds_train=ds_tr_clf, ds_val=ds_val_clf,
                     feature_cols=cols, encoder=enc)

    tr_act, val_act = tr[tr[target_col] > 0], val[val[target_col] > 0]
    ds_tr_reg = lgb.Dataset(tr_act[cols], label=tr_act[target_col],
                             categorical_feature=cat_idx, params=ds_params)
    ds_val_reg = lgb.Dataset(val_act[cols], label=val_act[target_col], categorical_feature=cat_idx,
                              reference=ds_tr_reg, params=ds_params)
    reg_data = dict(train=tr_act, val=val_act, test=test, ds_train=ds_tr_reg, ds_val=ds_val_reg,
                     feature_cols=cols, encoder=enc)

    return dict(clf=clf_data, reg=reg_data, test=test, feature_cols=cols, encoder=enc)


# Fixed default hyperparameters on both sides (no tune_classifier/
# tune_regressor_conditional random search) -- isolates the split as the only
# variable between "before" and "after" instead of conflating it with
# per-run tuning variance.
def run_hurdle_default(bundle, target_col):
    clf = train.train_classifier(bundle["clf"])
    reg = train.train_regressor_conditional(bundle["reg"])
    model = train.HurdleModel(clf, reg)
    test = bundle["test"]
    metrics = evaluate.score(model, test[bundle["feature_cols"]], test[target_col])
    return model, metrics


In [14]:
HONEST_CUTOFF = "2026-03-30"  # same cutoff w05_model_temp.ipynb uses

before_after = {}
for target_col, raw_feature_col in [
    ("target_gsc_clicks_30d", "f_gsc_clicks_90d"),
    ("target_ga4_sessions_30d", "f_ga4_sessions_90d"),
]:
    # before: naive random split, ignoring time and client
    tr_idx, val_idx, te_idx = naive_random_split(df, seed=0)
    naive_bundle = build_hurdle_from_idx(df, target_col, tr_idx, val_idx, te_idx)
    _, naive_metrics = run_hurdle_default(naive_bundle, target_col)
    del naive_bundle
    gc.collect()

    # after: honest split, same model/hyperparams
    splits = train.temporal_split(df, cutoff=HONEST_CUTOFF)
    honest_bundle = train.make_hurdle_datasets(df, target_col, splits)
    _, honest_metrics = run_hurdle_default(honest_bundle, target_col)
    del honest_bundle
    gc.collect()

    comparison = pd.DataFrame({
        "naive random split (before)": naive_metrics,
        "honest temporal split, gap_days=30 (after)": honest_metrics,
    }).T
    before_after[target_col] = comparison
    print(f"{target_col} -- hurdle model, before vs after")
    display(comparison)


target_gsc_clicks_30d -- hurdle model, before vs after


,n,mae,rmse,spearman,tweedie_deviance,mean_actual,mean_pred
naive random split (before),1133221.0,1.037671,8.804360,0.667641,1.103338,2.615701,2.558883
"honest temporal split, gap_days=30 (after)",1463957.0,1.279150,14.433681,0.567773,7.268792,1.926367,2.119981


target_ga4_sessions_30d -- hurdle model, before vs after


,n,mae,rmse,spearman,tweedie_deviance,mean_actual,mean_pred
naive random split (before),725227.0,2.699197,86.750455,0.724924,1.930785,4.910276,4.494074
"honest temporal split, gap_days=30 (after)",1137955.0,4.254128,114.323756,0.663050,12.010092,5.993525,3.433122


**Reading the gap**: content is anchored weekly, so a train row's 30-day-forward target window and a nearby row's 90-day trailing feature window routinely cover overlapping calendar days for the *same* `content_hash_id` which is fine when both rows land in train, but it becomes a leakage path once a naive split scatters those rows across train and test at random.

Worse, under a naive split a `content_hash_id`'s rows land in both train and test with no time ordering at all, so the model can end up training on a later week for a piece of content and being "tested" on an earlier week of the *same* content it already partially learned from `client_target_enc`.

`train.temporal_split`'s `gap_days=30` exists specifically to guarantee no train row's target window can extend into the test period at all, this is quantified directly in Section 3 below.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [15]:
AUDIT_CUTOFF = "2026-03-30"
CORR_FLAG_THRESHOLD = 0.95


def train_target_overlap_with_test(df, gap_days):
    """How many TRAIN rows have a target window (anchor_date, anchor_date+30]
    that extends past the point test starts? Zero is required for an honest
    split -- otherwise a training label is partly computed from days that are
    also inside the test evaluation period."""
    splits = train.temporal_split(df, cutoff=AUDIT_CUTOFF, gap_days=gap_days)
    train_rows = df.loc[df.index.intersection(splits.train_idx), ["anchor_date"]]
    target_window_end = train_rows["anchor_date"] + pd.Timedelta(days=30)
    contaminated = target_window_end > splits.test_start
    return int(contaminated.sum()), len(train_rows)


def encoder_leak_delta(df, target_col, splits):
    """How much would client_target_enc change if val rows had leaked into the
    fit? Refits on train+val combined and diffs against the train-only
    encoder make_hurdle_datasets actually uses -- confirms that fit boundary
    is load-bearing, not incidental."""
    d = train.prepare_target_dataset(df, target_col)
    train_all = d.loc[d.index.intersection(splits.train_idx)]
    val_start = splits.cutoff - pd.Timedelta(weeks=8)
    tr = train_all[train_all["anchor_date"] < val_start]
    val = train_all[train_all["anchor_date"] >= val_start]

    enc_train_only = train.ClientTargetEncoder().fit(tr["client_hash_id"], tr[target_col])
    enc_leaked = train.ClientTargetEncoder().fit(
        pd.concat([tr["client_hash_id"], val["client_hash_id"]]),
        pd.concat([tr[target_col], val[target_col]]),
    )
    shared_clients = enc_train_only.map_.index.intersection(enc_leaked.map_.index)
    diff = (enc_train_only.map_[shared_clients] - enc_leaked.map_[shared_clients]).abs()
    return dict(
        n_clients_compared=len(shared_clients),
        n_clients_changed=int((diff > 1e-9).sum()),
        max_abs_diff=float(diff.max()),
        mean_abs_diff=float(diff.mean()),
    )


n_overlap_honest, n_train_honest = train_target_overlap_with_test(df, gap_days=30)
n_overlap_naive, n_train_naive = train_target_overlap_with_test(df, gap_days=0)
print("train rows whose target window extends into the test period:")
print(f"  gap_days=30 (what temporal_split actually uses): {n_overlap_honest:,}/{n_train_honest:,}")
print(f"  gap_days=0  (what a naive cutoff-only split would do): {n_overlap_naive:,}/{n_train_naive:,}")
print()

for target_col in ["target_gsc_clicks_30d", "target_ga4_sessions_30d"]:
    print(f"=== {target_col} ===")
    splits = train.temporal_split(df, cutoff=AUDIT_CUTOFF)
    data = train.make_hurdle_datasets(df, target_col, splits)
    tr = data["clf"]["train"]

    numeric_cols = [c for c in data["feature_cols"] if c not in train.CATEGORICAL_FEATURES]
    corrs = tr[numeric_cols + [target_col]].corr(numeric_only=True)[target_col].drop(target_col)
    flagged = corrs[corrs.abs() > CORR_FLAG_THRESHOLD]
    print(f"correlation with target (train only), flagged |corr| > {CORR_FLAG_THRESHOLD}:")
    display(flagged.sort_values(key=abs, ascending=False) if len(flagged) else "none flagged")

    print("client_target_enc leak-if-fit-on-train+val delta:", encoder_leak_delta(df, target_col, splits))
    print()


train rows whose target window extends into the test period:
  gap_days=30 (what temporal_split actually uses): 0/3,216,594
  gap_days=0  (what a naive cutoff-only split would do): 1,074,629/3,216,594

=== target_gsc_clicks_30d ===
correlation with target (train only), flagged |corr| > 0.95:


'none flagged'

client_target_enc leak-if-fit-on-train+val delta: {'n_clients_compared': 41, 'n_clients_changed': 41, 'max_abs_diff': 2.09322996590452, 'mean_abs_diff': 0.23966535682838497}

=== target_ga4_sessions_30d ===
correlation with target (train only), flagged |corr| > 0.95:


'none flagged'

client_target_enc leak-if-fit-on-train+val delta: {'n_clients_compared': 15, 'n_clients_changed': 15, 'max_abs_diff': 14.882847770239326, 'mean_abs_diff': 2.1431225793402273}



**Feature-set safety summary**

| Feature group | Example columns | Why it's safe as of `anchor_date` |
|---|---|---|
| Raw 90d trailing aggregates | `f_gsc_clicks_90d`, `f_ga4_sessions_90d`, `f_sessions_*_90d` | Calendar-day rolling window strictly preceding `anchor_date` by construction in `build_windows.py` |
| Derived ratios/shares | `*_share`, `f_gsc_ctr_90d`, `f_gsc_momentum_ratio` | Pure functions of the 90d aggregates above; inherit their safety. |
| Categorical content metadata | `content_type`, `main_intent`, `competition_level` | Descriptive attributes of the content itself, not activity-derived. |
| `client_target_enc` | — | Confirmed above: fit exclusively on the train split's client stats; val/test only ever call `.transform`, which just looks up the precomputed map. |

**Open questions this notebook can't verify on its own** (would need checking against the pipeline, not this audit): whether `word_count`, `search_volume`, and `competition` are captured as-of `anchor_date` or reflect live values regardless of anchor. If its the latter, they'd technically use information not yet available at the historical prediction points used for evaluation.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (w05_model_temp.ipynb, closing summary):**

> "for GA4 sessions, the hurdle model is a clean win over naive across MAE/RMSE/Spearman, and the client-holdout gap is small — mostly transferable signal, not client memorization."

**Rewritten:**

On the 2026-03-30 temporal test split, the GA4 hurdle model scored lower MAE and RMSE and higher Spearman correlation than the naive scaled-baseline. On a 20-client holdout sample, the same model's error on those never-seen clients was close to its error on seen clients evaluated in the same period, which is consistent with the model relying more on transferable activity patterns than on memorized client identity.

Though this is one holdout draw and one time cutoff, not a general property established across clients or time periods. Treat it as a reason to prioritize the GA4 hurdle model over the naive baseline for this test window, not as a claim that it will generalize equally well to a different client base or a different point in time.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.